# 데모 전용 노트북: 학습 없이 채팅 데모 실행

`train_and_eval_colab.ipynb`로 학습을 1회 완료한 뒤, 시연할 때마다 이 노트북만 실행한다.

- 학습된 LoRA 어댑터를 Google Drive(`MyDrive/lora-kifrs-adapter`)에서 로드
- RAG 인덱스 재구축 후 Gradio 채팅 데모 실행
- 소요 시간 약 10분 (학습 1시간 불필요)

**환경**: T4 GPU로도 충분 (추론만 하므로 A100 불필요)

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate faiss-cpu sentence-transformers==3.3.1 "gradio==4.44.1" requests

## 1. 데이터 준비 (RAG 코퍼스용)

In [ ]:
REPO_URL = "https://github.com/gyeongmin100/kifrs-qa-llm.git"
!git clone {REPO_URL} repo
%cd repo
!python src/scrape.py --rows 200 --sleep 0.3
!python src/prepare_data.py

In [ ]:
import json

rag_corpus = [json.loads(l) for l in open("data/processed/rag_corpus.jsonl", encoding="utf-8")]
print("RAG 코퍼스:", len(rag_corpus), "건")

## 2. 모델 + 학습된 어댑터 로드 (Drive)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ADAPTER_DIR = "/content/drive/MyDrive/lora-kifrs-adapter"
!ls {ADAPTER_DIR}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
SYSTEM_PROMPT = (
    "당신은 한국 회계기준(K-IFRS, 일반기업회계기준) 전문 회계 자문가입니다. "
    "질의된 회계처리 사안에 대해 관련 회계기준과 문단을 인용하며 명확한 결론을 제시하세요."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
lora_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
lora_model.eval()
print("어댑터 로드 완료")

## 3. RAG 인덱스

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# bge-m3 임베딩 (FlagEmbedding은 transformers 4.46과 버전 충돌이 있어 sentence-transformers 사용)
embedder = SentenceTransformer("BAAI/bge-m3", model_kwargs={"torch_dtype": torch.float16})

corpus_texts = [f"{r['title']}\n{r['question']}" for r in rag_corpus]
emb = embedder.encode(corpus_texts, batch_size=64, convert_to_numpy=True, show_progress_bar=True).astype("float32")
faiss.normalize_L2(emb)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

def retrieve(question, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    return [(rag_corpus[i], float(s)) for i, s in zip(idxs[0], scores[0])]

print("인덱스 크기:", index.ntotal)

## 4. 생성 함수

In [ ]:
RAG_TEMPLATE = (
    "다음은 과거 유사한 질의회신 사례입니다.\n\n{context}\n\n"
    "위 사례를 참고하여 아래 질의에 답변하세요.\n\n[질의]\n{question}"
)

def build_prompt(question, use_rag=False):
    if not use_rag:
        return question
    docs = retrieve(question, k=3)
    context = "\n\n".join(
        f"[사례 {i+1}] {d['title']}\n질의: {d['question'][:500]}\n회신: {d['answer'][:500]}"
        for i, (d, _) in enumerate(docs)
    )
    return RAG_TEMPLATE.format(context=context, question=question)

@torch.no_grad()
def generate(model, question, use_rag=False, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, use_rag)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def generate_config(question, config):
    """config: 'base' | 'rag' | 'lora' | 'lora_rag'"""
    use_rag = config in ("rag", "lora_rag")
    if config in ("base", "rag"):
        with lora_model.disable_adapter():
            return generate(lora_model, question, use_rag)
    return generate(lora_model, question, use_rag)

## 5. Gradio 채팅 데모
실행하면 공개 URL(`https://xxx.gradio.live`)이 출력된다. 72시간 유효.

In [ ]:
import gradio as gr

def chat_fn(message, history, config):
    answer = generate_config(message, config)
    if config in ("rag", "lora_rag"):
        docs = retrieve(message, k=3)
        refs = "\n".join(
            f"- [{d['docNumber']}] {d['title']} (유사도 {s:.2f})" for d, s in docs
        )
        answer += f"\n\n---\n**참고한 질의회신 사례**\n{refs}"
    return answer

demo = gr.ChatInterface(
    chat_fn,
    additional_inputs=[
        gr.Dropdown(["lora_rag", "lora", "rag", "base"], value="lora_rag", label="모델 구성")
    ],
    title="회계기준 질의회신 어시스턴트",
    description="K-IFRS·일반기업회계기준 질의를 입력하세요. LoRA 파인튜닝 + 유사 사례 RAG 기반으로 답변합니다.",
    examples=[
        ["회사가 보유한 자기주식을 소각하는 경우 회계처리는 어떻게 하나요?"],
        ["리스기간이 12개월 이하인 사무실 임차 계약도 사용권자산을 인식해야 하나요?"],
    ],
)
demo.launch(share=True)